# ***Notebook For Standardizing For Logistic DB Tables***

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
import os
from pathlib import Path

from sqlalchemy import create_engine, text

import warnings
warnings.filterwarnings('ignore')

In [2]:
env_path = Path.cwd()/"github_repo_cloned"/"the_distribution_hub_optimizer"/".env"
load_dotenv(env_path)


host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
paswd = os.getenv("DB_PASSWORD")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")
output_dir = Path(os.getenv("OUTPUT_DIR", Path.cwd()/"outputs"))
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
engine = create_engine(f"mysql+pymysql://{user}:{paswd}@{host}:{port}/{db}")


def sql(query, engine = engine):
    with engine.connect() as conn:
        df = pd.read_sql(text(query),conn)
    return df

In [4]:
bill = sql('select * from bill') 

In [5]:
gsdp = sql('select * from gsdp') 

In [6]:
population = sql('select * from population')

## **Data Standardization**

### **Initial Integrity Check - Obvious Errors**

In [7]:
bill.head()

,state_code,state_name,year,month,intra_suplliers,intra_bill,intra_assets,inter_out_suplliers,inter_out_bill,inter_out_asset,inter_in_suplliers,inter_in_bill,inter_in_asset
0,1,JAMMU AND KASHMIR,2018,7,3171,120989,2083.7662087950002,1711,25744,2243.508837,15449,151999,4567.124618
1,1,JAMMU AND KASHMIR,2018,8,3292,126112,2245.7754784900003,1996,30376,2535.885482,17023,180173,5157.642880
2,1,JAMMU AND KASHMIR,2018,9,3484,130396,2387.847554812,2114,30149,2215.296839,16962,186069,5240.539348
3,1,JAMMU AND KASHMIR,2018,10,3784,135435,2545.633385386,2273,32807,2160.836647,17722,202717,5138.989032
4,1,JAMMU AND KASHMIR,2018,11,3851,123745,2178.749353559,2080,26704,1751.514975,16757,165151,4435.449635


In [8]:
gsdp.head()

,state/ut,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24
0,Andhra Pradesh,873721.106301,925839.118118,978581.462125,1.148471e+06,1.303524e+06,1439673.87
1,Arunachal Pradesh,25334.870000,30023.650000,30525.350000,3.477476e+04,3.962996e+04,43991.61
2,Assam,309336.320000,346850.680000,339802.980000,4.114538e+05,4.931666e+05,565400.58
3,Bihar,527975.819958,581855.484686,567262.504733,6.503024e+05,7.513956e+05,854429.00
4,Chhattisgarh,327106.660000,344672.040000,352327.510000,4.105249e+05,4.643990e+05,505886.51


In [9]:
population.head()

,state/ut,2018,2019,2020,2021,2022,2023,2024,2025,2026
0,Jammu & Kashmir*(UT),13101,13203,13305,13408,13505,13603,13701,13798,13896
1,Himachal Pradesh,7253,7300,7347,7394,7431,7468,7505,7542,7579
2,Punjab,29619,29859,30099,30339,30535,30730,30926,31122,31318
3,Haryana,28266,28672,29077,29483,29846,30209,30573,30936,31299
4,NCT of Delhi*,19435,19814,20193,20571,20965,21359,21752,22146,22540


```Bill Table```

In [10]:
bill.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3664 entries, 0 to 3663
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   state_code           3664 non-null   int64  
 1   state_name           3664 non-null   object 
 2   year                 3664 non-null   int64  
 3   month                3664 non-null   int64  
 4   intra_suplliers      3664 non-null   object 
 5   intra_bill           3664 non-null   object 
 6   intra_assets         3664 non-null   object 
 7   inter_out_suplliers  3664 non-null   int64  
 8   inter_out_bill       3664 non-null   int64  
 9   inter_out_asset      3664 non-null   float64
 10  inter_in_suplliers   3664 non-null   int64  
 11  inter_in_bill        3664 non-null   int64  
 12  inter_in_asset       3664 non-null   float64
dtypes: float64(2), int64(7), object(4)
memory usage: 372.3+ KB


In [24]:
bill.select_dtypes(include='number').agg(['mean', 'median', 'std', 'min', 'max']).T.round(2)

,mean,median,std,min,max
state_code,21.15,19.00,16.35,1.00,97.00
year,2022.06,2022.00,2.36,2018.00,2026.00
month,6.50,7.00,3.44,1.00,12.00
inter_out_suplliers,25316.82,6697.00,37773.49,0.00,168901.00
inter_out_bill,771196.64,195438.00,1336783.14,0.00,8649985.00
inter_out_asset,26511.34,9506.45,39191.20,0.00,254765.99
inter_in_suplliers,47376.21,24052.50,56359.00,5.00,259235.00
inter_in_bill,768406.13,356606.50,987534.15,20.00,5399958.00
inter_in_asset,27908.82,12866.70,37841.94,0.38,259141.83
